# HEN Colab Runner

This notebook runs the HEN project end to end in Google Colab: clone the repo, install dependencies, download the 27-class ImageNet subset, train a compact Joint HEN model, and evaluate it.

Use **Runtime > Change runtime type > GPU** before running. The default `FAST_MODE=True` is a small smoke test so new readers can confirm the pipeline works. Set `FAST_MODE=False` to run the full 27-class experiment configuration.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HengchunSong/HEN.git"
PROJECT_DIR = Path("/content/HEN")

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

In [ ]:
!pip install -q datasets huggingface_hub pyarrow fsspec pandas matplotlib

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Configuration

`FAST_MODE=True` downloads a tiny balanced subset and trains briefly. It is meant to prove that everything runs. For the full experiment, set `FAST_MODE=False`; that uses the full downloaded subset, `MobileNetV3-Large`, and a longer schedule.

In [ ]:
FAST_MODE = True

if FAST_MODE:
    MAX_PER_CLASS = 20
    BACKBONE = "mobilenet_v3_small"
    EPOCHS = 2
    BATCH_SIZE = 64
    IMAGE_SIZE = 160
    RUN_NAME = "colab_smoke_joint_hen"
else:
    MAX_PER_CLASS = None
    BACKBONE = "mobilenet_v3_large"
    EPOCHS = 25
    BATCH_SIZE = 96
    IMAGE_SIZE = 224
    RUN_NAME = "colab_full_joint_hen_mobilenet_v3_large"

DATA_ROOT = Path("data/imagenet_subset_food_colab_smoke" if FAST_MODE else "data/imagenet_subset_food")
OUTPUT_DIR = Path("outputs") / RUN_NAME

print({
    "FAST_MODE": FAST_MODE,
    "DATA_ROOT": str(DATA_ROOT),
    "MAX_PER_CLASS": MAX_PER_CLASS,
    "BACKBONE": BACKBONE,
    "EPOCHS": EPOCHS,
    "BATCH_SIZE": BATCH_SIZE,
    "IMAGE_SIZE": IMAGE_SIZE,
    "OUTPUT_DIR": str(OUTPUT_DIR),
})

## Download the 27-class subset

The downloader streams from the Hugging Face dataset mirror named in `configs/tree_27cls_food.json`. The smoke test is intentionally small. The full run downloads the complete balanced subset used in the report.

In [ ]:
cmd = [
    "python", "download_imagenet_27.py",
    "--config", "configs/tree_27cls_food.json",
    "--output-root", str(DATA_ROOT),
]
if MAX_PER_CLASS is not None:
    cmd += ["--max-per-class", str(MAX_PER_CLASS)]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd

train_manifest = DATA_ROOT / "manifests" / "train.csv"
val_manifest = DATA_ROOT / "manifests" / "val.csv"
train_df = pd.read_csv(train_manifest)
val_df = pd.read_csv(val_manifest)

print("Train images:", len(train_df))
print("Val images:", len(val_df))
display(train_df.groupby(["level1_name", "level2_name", "leaf_name"]).size().rename("train_count").reset_index().head(12))

## Train compact Joint HEN

This is the compact HEN family used for the resource-efficiency comparison. The full configuration uses `MobileNetV3-Large` and hierarchical losses for top, mid, and leaf predictions.

In [ ]:
train_cmd = [
    "python", "train_joint_hen.py",
    "--data-root", str(DATA_ROOT),
    "--backbone", BACKBONE,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--image-size", str(IMAGE_SIZE),
    "--num-workers", "2",
    "--channels-last",
    "--output-dir", str(OUTPUT_DIR),
]

print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)

## Evaluate

In [ ]:
eval_path = OUTPUT_DIR / "eval_summary.json"
eval_cmd = [
    "python", "evaluate_joint_hen.py",
    "--run-dir", str(OUTPUT_DIR),
    "--data-root", str(DATA_ROOT),
    "--output-path", str(eval_path),
    "--batch-size", str(max(BATCH_SIZE, 128)),
    "--num-workers", "2",
]

print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

In [ ]:
import json

summary = json.loads((OUTPUT_DIR / "summary.json").read_text())
eval_summary = json.loads(eval_path.read_text())

print("Training summary")
print(json.dumps({
    "backbone": summary.get("backbone"),
    "best_leaf_val_acc_pct": summary.get("best_leaf_val_acc_pct"),
    "epochs": summary.get("epochs"),
}, indent=2))

print("\nEvaluation summary")
print(json.dumps({
    "top_acc": round(eval_summary["top_acc"] * 100, 2),
    "mid_acc": round(eval_summary["mid_acc"] * 100, 2),
    "leaf_acc": round(eval_summary["leaf_acc"] * 100, 2),
}, indent=2))

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_json(OUTPUT_DIR / "history.json")
plt.figure(figsize=(8, 4))
plt.plot(history["epoch"], history["val_level1_acc"] * 100, label="Top")
plt.plot(history["epoch"], history["val_level2_acc"] * 100, label="Mid")
plt.plot(history["epoch"], history["val_leaf_acc"] * 100, label="Leaf")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy (%)")
plt.title("Joint HEN validation accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Save outputs

The cell below creates a zip archive with the trained checkpoint and JSON summaries. Download it from the Colab file browser or mount Google Drive before running a full experiment.

In [ ]:
archive_base = Path("/content") / RUN_NAME
!zip -qr "{archive_base}.zip" "{OUTPUT_DIR}"
print("Created:", f"{archive_base}.zip")